In [ ]:
from wotan import flatten
import matplotlib.pyplot as plt
import pandas as pd
from astropy.timeseries import LombScargle
import numpy as np
from astropy.timeseries import BoxLeastSquares

In [ ]:
# cargar datos
file = pd.read_csv("tic_72829000_s_98_corrected.csv")

#print(file.columns)
time = file["time"]
flux = file["flux"]
print(time,flux)

# plot 
plt.scatter(time, flux, s=5)

In [ ]:
def explore_detrend(window_length, break_tol):

    flat, trend = flatten(
        time,
        flux,
        window_length=window_length,
        break_tolerance=break_tol,
        method="hspline",
        return_trend=True
    )

    plt.figure(figsize=(12,5))
    plt.scatter(time, flux, s=3, color='k', label="Original")
    plt.plot(time, trend, color='r', label="Trend")
    plt.legend()
    plt.title(f"window={window_length}, break={break_tol}")
    plt.show()

    return flat, trend

In [ ]:
flat_flux, trend_flux = explore_detrend(.65, 1.0)

df = pd.DataFrame({
    "time": time,          
    "flat_flux": flat_flux,
    "trend_flux": trend_flux
})
t0_B = time[np.argmin(flat_flux)]
print(t0_B)
df.to_csv("TIC_199780530_s_6_trending.csv", index=False)

In [ ]:

def iterative_transit_search(time, flux, max_signals=5):

    residuals = flux.copy()
    detected = []

    for n in range(max_signals):

        print(f"\n--- Buscando señal {n+1} ---")

        # quitar NaNs
        mask_valid = np.isfinite(residuals)
        t_clean = time[mask_valid]
        f_clean = residuals[mask_valid]

        bls = BoxLeastSquares(t_clean, f_clean)

        periods = np.linspace(0.5, 20, 10000)
        duration = 0.1

        power = bls.power(periods, duration)

        best_idx = np.argmax(power.power)
        P = periods[best_idx]
        t0 = power.transit_time[best_idx]

        print(f"Periodo detectado: {P:.4f} d")

        # =========================
        #  evitar duplicados
        # =========================
        if any(abs(P - p)/p < 0.05 for p, _ in detected):
            print("Periodo ya detectado, saltando...")
            continue

        # =========================
        # detectar alias P/2
        # =========================
        if any(abs(P - 2*p)/p < 0.05 or abs(2*P - p)/p < 0.05 for p, _ in detected):
            print("⚠️ Posible alias (P vs P/2)")

        # =========================
        # modelo box
        # =========================
        model = bls.model(time, P, duration, t0)

        # =========================
        # masking 
        # =========================
        in_transit = model < (np.median(model) - 1e-4)

        residuals = residuals.copy()
        residuals[in_transit] = np.nan

        # guardar
        detected.append((P, t0))

        print(f"Señales detectadas hasta ahora: {len(detected)}")

    return detected, residuals

In [ ]:
detected, residuals = iterative_transit_search(time, flux)
P_detected = detected[0][0]# estamos haciendo un diccionario, entonces primerp cuargamos en detected dos valores, pero a su vez el return nos da dos entonces accedemos a detected con [0]y luego a la posicion en que se guarda todo en detected
t0_detected = detected[0][1]

print("P_detected:", P_detected)

In [ ]:
def plot_phase(time, flux, P):

    # 🔹 fase centrada en primario
    t0 = time[np.argmin(flux)]
    phase = ((time - t0) % P) / P - 0.5

    # 🔹 duplicar (evita cortes)
    phase = np.concatenate([phase, phase + 1])
    flux = np.concatenate([flux, flux])

    # 🔹 primario
    i1 = np.argmin(flux)
    p1, f1 = phase[i1], flux[i1]

    # 🔹 secundario (lejos del primario)
    mask = np.abs(phase - p1) > 0.2
    i2 = np.argmin(flux[mask])
    p2, f2 = phase[mask][i2], flux[mask][i2]

    # 🔹 baseline
    baseline = np.median(flux)

    # 🔹 plot
    plt.figure(figsize=(10,4))
    plt.scatter(phase, flux, s=3, color='k')

    # marcar eclipses
    plt.axvline(p1, color='red', ls='--')
    plt.axvline(p2, color='blue', ls='--')

    # profundidades (opcional pero útil)
    plt.vlines(p1, f1, baseline, color='red')
    plt.vlines(p2, f2, baseline, color='blue')

    plt.xlim(-0.5, 1.5)
    plt.xlabel("Phase")
    plt.ylabel("Flux")
    plt.title(f"P = {P:.4f}")
    plt.grid(alpha=0.3)
    
P_real = P_detected
plot_phase(time, flat_flux, P_real*2)

In [ ]:
def get_rotation_period(time, flux_flat, P_orb,
                        eclipse_width=0.08,
                        min_period=0.1,
                        max_period=20,
                        plot=True):

    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.timeseries import LombScargle

    # =========================
    # 🔹 asegurar numpy arrays
    # =========================
    time = np.asarray(time)
    flux_flat = np.asarray(flux_flat)

    # =========================
    # 🔹 fase orbital (para masking)
    # =========================
    t0 = time[np.argmin(flux_flat)]
    phase = ((time - t0) % P_orb) / P_orb - 0.5

    # =========================
    # 🔹 máscara de eclipses
    # =========================
    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # =========================
    # 🔹 datos limpios
    # =========================
    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    mask_valid = np.isfinite(flux_clean)
    time_clean = time_clean[mask_valid]
    flux_clean = flux_clean[mask_valid]

    # =========================
    # 🔹 normalizar
    # =========================
    flux_clean = flux_clean / np.median(flux_clean)

    # =========================
    # 🔹 Lomb-Scargle
    # =========================
    freq, power = LombScargle(time_clean, flux_clean).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period
    )

    periods = 1 / freq

    # =========================
    # 🔹 periodo candidato
    # =========================
    Prot = periods[np.argmax(power)]

    # =========================
    # 🔥 chequeo simple P vs 2P
    # =========================
    def scatter_metric(P):
        ph = (time_clean % P) / P
        return np.std(ph)

    if scatter_metric(2 * Prot) < scatter_metric(Prot):
        print("⚠️ Usando 2P (alias detectado)")
        Prot = 2 * Prot

    # =========================
    # 🔹 plots
    # =========================
    if plot:

        # -------- periodograma --------
        plt.figure(figsize=(8,4))
        plt.plot(periods, power, color='k')
        plt.axvline(Prot, color='red', ls='--')
        plt.xlabel("Period [days]")
        plt.ylabel("Power")
        plt.title("Rotation Period")
        plt.xlim(0, min(max_period, 10))
        plt.grid(alpha=0.3)

        # -------- FASE ROTACIÓN --------
        phase_rot = (time_clean % Prot) / Prot - 0.5
        phase_rot_plot = np.concatenate([phase_rot, phase_rot + 1])
        flux_rot_plot = np.concatenate([flux_clean, flux_clean])

        # -------- FASE ORBITAL --------
        phase_orb = ((time - t0) % P_orb) / P_orb - 0.5
        phase_orb_plot = np.concatenate([phase_orb, phase_orb + 1])
        flux_orb_plot = np.concatenate([flux_flat, flux_flat])

        # -------- PLOT FUSIONADO --------
        plt.figure(figsize=(9,5))

        # orbital (fondo)
        plt.scatter(
            phase_orb_plot,
            flux_orb_plot,
            s=5,
            alpha=0.25,
            color='C1',
            label=f"Orbital (P={P_orb:.3f} d)"
        )

        # rotación (encima)
        plt.scatter(
            phase_rot_plot,
            flux_rot_plot,
            s=5,
            alpha=0.5,
            color='C0',
            label=f"Rotation (P={Prot:.3f} d)"
        )

        plt.xlabel("Folded Phase")
        plt.ylabel("Normalized Flux")
        plt.title("Orbital vs Rotational Modulation")

        plt.xlim(-0.5, 1.5)

        # límites robustos
        all_flux = np.concatenate([flux_rot_plot, flux_orb_plot])
        plt.ylim(np.percentile(all_flux, 1),
                 np.percentile(all_flux, 99))

        plt.grid(alpha=0.3)
        plt.legend()

    # =========================
    # 🔹 output
    # =========================
    print("\n--- RESULTADOS ---")
    print(f"P_orb = {P_orb:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_orb:.3f}")

    return Prot, periods, power, time_clean, flux_clean

In [ ]:
P_real = P_detected

Prot, periods, power, time_clean, flux_clean = get_rotation_period(
    time,
    flat_flux,
    P_real,
    plot=True
)

In [ ]:
def compute_width(p_center):

    if p_center is None:
        return None, None, None

    # distancia circular
    dist = np.abs(phase - p_center)
    dist = np.minimum(dist, 1 - dist)

    mask = dist < 0.25
    ph = phase[mask]
    fl = flux[mask]

    # centrar en 0
    ph = ph - p_center
    ph[ph > 0.5] -= 1
    ph[ph < -0.5] += 1

    # grid de anchos
    widths = np.linspace(0.01, 0.25, 150)

    best_chi2 = np.inf
    best = None

    for w in widths:

        inside = np.abs(ph) < w/2
        outside = ~inside

        if np.sum(inside) < 5 or np.sum(outside) < 5:
            continue

        f_in = np.mean(fl[inside])
        f_out = np.mean(fl[outside])

        model = np.where(inside, f_in, f_out)

        chi2 = np.sum((fl - model)**2)

        if chi2 < best_chi2:
            best_chi2 = chi2
            best = (w, f_in, f_out)

    if best is None:
        return None, None, None

    w_best, f_in, f_out = best

    phi_start = p_center - w_best/2
    phi_end   = p_center + w_best/2

    return w_best, phi_start, phi_end

In [ ]:
results = analyze_eclipses(time, flat_flux, P_real*2)